# Thyroid Histopathology YOLOv8m with LocaQ Loss

Run the configuration cell first. Set `DATA_YAML` to your dataset YAML and choose `OUTPUT_ROOT`. GPU 0 is the original experiment device. To evaluate an existing checkpoint, set `CHECKPOINT_OVERRIDE` and skip the training cell. Start Jupyter from the repository root.

Any saved output reflects the original run and may contain its original paths.


In [ ]:
from pathlib import Path

# Set this to the YAML for the corresponding paper dataset and splits.
DATA_YAML = str(Path("datasets/thyroid_histopathology/data.yaml").expanduser().resolve())
OUTPUT_ROOT = Path("runs/thyroid_histopathology_yolov8m_locaq").expanduser().resolve()

# Optional: point this at an existing checkpoint to run evaluation only.
CHECKPOINT_OVERRIDE = None
RUN_NAME = "QALoss_YOLOv8m"

PROJECT_DIR = str(OUTPUT_ROOT / "training")
MODEL_PATH = (
    str(Path(CHECKPOINT_OVERRIDE).expanduser().resolve())
    if CHECKPOINT_OVERRIDE is not None
    else str(Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt")
)


In [1]:
import inspect
from locaq_loss import QALoss as LocaQLoss

source = inspect.getsource(LocaQLoss)
assert "_scale_norm" not in source, "BUG: _scale_norm found. File is wrong."
print("locaq_loss.py verified clean.")

qa_loss.py verified clean.


In [2]:
# ============================================================
# LocaQLoss YOLOv8m TRAINING — Thyroid Histopathology
# Complete script — just run this
# ============================================================

import types
import sys
import os
import random
import numpy as np
import torch
from ultralytics import YOLO
from locaq_loss import QALoss as LocaQLoss

# ============================================================
# VERIFY CORRECT locaq_loss.py IS LOADED
# ============================================================

import inspect
source = inspect.getsource(LocaQLoss)
if "get_assigned_targets_and_loss" in source:
    print("Correct locaq_loss.py loaded (8.4.x compatible).")
else:
    raise RuntimeError("OLD locaq_loss.py loaded. Replace the file and restart kernel.")

# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"]          = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic    = True
torch.backends.cudnn.benchmark        = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32       = False
torch.use_deterministic_algorithms(True)

# ============================================================
# PATHS
# ============================================================


# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO("yolov8m.yaml")

# ============================================================
# INJECT LocaQLoss VIA on_train_start
# Overrides trainer.model.init_criterion AFTER trainer
# creates its own model — this is the correct timing
# ============================================================

def inject_locaq_loss(trainer):
    trainer.model.init_criterion = types.MethodType(
        lambda self: LocaQLoss(
            self,
            beta=1.0,
            vfl_alpha=0.75,
            vfl_gamma=2.0,
            anchor_area=1024.0,
            fn_clamp_max=3.0
        ),
        trainer.model
    )
    print(f"[LocaQLoss] Injected into trainer.model successfully.")

model.add_callback("on_train_start", inject_locaq_loss)

# ============================================================
# TRAIN
# ============================================================

model.train(
    data=DATA_YAML,
    epochs=250,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=0.0002,
    lrf=0.01,
    weight_decay=0.01,
    cos_lr=True,
    warmup_epochs=5,
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.25,
    degrees=10,
    translate=0.05,
    scale=0.25,
    shear=1.0,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.4,
    close_mosaic=20,
    mixup=0.0,
    seed=SEED,
    deterministic=True,
    workers=0,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    save_period=10,
    device=0
)

Correct qa_loss.py loaded (8.4.x compatible).
Ultralytics 8.4.51  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=E:\Mahin\Thyroid Project\Thyroid_Dataset\data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=250, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.35, hsv_v=0.25, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.yaml, momentum=0.937, mosaic=0.4, multi_scale=0.0, name=QALoss_YOLOv8m, nbs=64, nms=Fa

c:\ProgramData\anaconda3\envs\yolo_gpu\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


AMP: checks passed 
train: Fast image access  (ping: 0.00.0 ms, read: 406.159.4 MB/s, size: 45.6 KB)
train: Scanning E:\Mahin\Thyroid Project\Thyroid_Dataset\final_3class_dataset\train\labels.cache... 327 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 327/327  0.0s
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.00.0 ms, read: 475.3180.2 MB/s, size: 52.0 KB)
val: Scanning E:\Mahin\Thyroid Project\Thyroid_Dataset\final_3class_dataset\val\labels.cache... 45 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 45/45  0.0s
optimizer: AdamW(lr=0.0002, momentum=0.937) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.01), 83 bias(decay=0.0)
Plotting labels to E:\Mahin\Thyroid Project\Thyroid_Dataset\Final_proposed_model\FN_ALoss_Project\models_result\QALoss_YOLOv8m\labels.jpg... 


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000251B0357970>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

In [4]:
# ============================================================
# COMPLETE TEST EVALUATION PIPELINE
# TEST RESULTS
# ============================================================

import os
import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ============================================================
# PATHS
# ============================================================



SAVE_DIR = str(OUTPUT_ROOT / "evaluation_1")

os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(MODEL_PATH)

# ============================================================
# RUN TEST VALIDATION
# ============================================================

metrics = model.val(
    data=DATA_YAML,
    split="test",

    imgsz=640,
    batch=8,

    conf=0.25,
    iou=0.5,

    device=0,

    save_json=True,
    plots=True,

    project=SAVE_DIR,
    name="test_eval"
)

# ============================================================
# CLASS NAMES
# ============================================================

class_names = model.names

num_classes = len(class_names)

print("\nClasses:")
print(class_names)

# ============================================================
# YOLO METRICS
# ============================================================

precision_per_class = metrics.box.p
recall_per_class = metrics.box.r

map50_per_class = metrics.box.ap50
map5095_per_class = metrics.box.ap

# ============================================================
# F1 SCORE
# ============================================================

f1_per_class = (
    2
    * precision_per_class
    * recall_per_class
) / (
    precision_per_class
    + recall_per_class
    + 1e-9
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = metrics.confusion_matrix.matrix

cm = cm[:num_classes, :num_classes]

# ============================================================
# SENSITIVITY / SPECIFICITY
# ============================================================

sensitivity_list = []
specificity_list = []

tp_list = []
fp_list = []
fn_list = []
tn_list = []

for i in range(num_classes):

    TP = cm[i, i]

    FN = cm[i, :].sum() - TP

    FP = cm[:, i].sum() - TP

    TN = cm.sum() - (TP + FN + FP)

    sensitivity = TP / (TP + FN + 1e-9)

    specificity = TN / (TN + FP + 1e-9)

    tp_list.append(TP)
    fp_list.append(FP)
    fn_list.append(FN)
    tn_list.append(TN)

    sensitivity_list.append(sensitivity)
    specificity_list.append(specificity)

# ============================================================
# MACRO METRICS
# ============================================================

macro_precision = np.mean(precision_per_class)
macro_recall = np.mean(recall_per_class)
macro_f1 = np.mean(f1_per_class)

macro_sensitivity = np.mean(sensitivity_list)
macro_specificity = np.mean(specificity_list)

macro_map50 = np.mean(map50_per_class)
macro_map5095 = np.mean(map5095_per_class)

# ============================================================
# RESULTS TABLE
# ============================================================

results_df = pd.DataFrame({

    "Class": [
        class_names[i]
        for i in range(num_classes)
    ],

    "Precision": precision_per_class,
    "Recall": recall_per_class,
    "F1-Score": f1_per_class,

    "Sensitivity": sensitivity_list,
    "Specificity": specificity_list,

    "mAP50": map50_per_class,
    "mAP50-95": map5095_per_class,

    "TP": tp_list,
    "FP": fp_list,
    "FN": fn_list,
    "TN": tn_list
})

results_df = results_df.round(4)

# ============================================================
# ADD MACRO ROW
# ============================================================

macro_row = pd.DataFrame([{

    "Class": "Macro Average",

    "Precision": round(macro_precision, 4),
    "Recall": round(macro_recall, 4),
    "F1-Score": round(macro_f1, 4),

    "Sensitivity": round(macro_sensitivity, 4),
    "Specificity": round(macro_specificity, 4),

    "mAP50": round(macro_map50, 4),
    "mAP50-95": round(macro_map5095, 4),

    "TP": np.sum(tp_list),
    "FP": np.sum(fp_list),
    "FN": np.sum(fn_list),
    "TN": np.sum(tn_list)
}])

results_df = pd.concat(
    [results_df, macro_row],
    ignore_index=True
)

# ============================================================
# SAVE CSV
# ============================================================

csv_path = os.path.join(
    SAVE_DIR,
    "classwise_metrics.csv"
)

results_df.to_csv(
    csv_path,
    index=False
)

print("\nSaved metrics CSV:")
print(csv_path)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n================================================")
print("TEST RESULTS")
print("================================================\n")

print(results_df)

# ============================================================
# CONFUSION MATRIX PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        class_names[i]
        for i in range(num_classes)
    ]
)

disp.plot(
    cmap="Blues",
    ax=ax,
    colorbar=False
)

plt.title("Detection Confusion Matrix")

cm_path = os.path.join(
    SAVE_DIR,
    "confusion_matrix.png"
)

plt.savefig(
    cm_path,
    dpi=600,
    bbox_inches="tight"
)

plt.close()

print("\nSaved confusion matrix.")

# ============================================================
# CLASS-WISE BAR PLOT
# ============================================================

metrics_to_plot = [
    "Precision",
    "Recall",
    "F1-Score",
    "Sensitivity",
    "Specificity",
    "mAP50",
    "mAP50-95"
]

for metric_name in metrics_to_plot:

    plt.figure(figsize=(12, 6))

    plt.bar(
        results_df["Class"][:-1],
        results_df[metric_name][:-1]
    )

    plt.xticks(rotation=30)

    plt.ylabel(metric_name)

    plt.title(f"Class-wise {metric_name}")

    save_path = os.path.join(
        SAVE_DIR,
        f"{metric_name}.png"
    )

    plt.savefig(
        save_path,
        dpi=600,
        bbox_inches="tight"
    )

    plt.close()

print("\nSaved class-wise metric plots.")

# ============================================================
# FALSE NEGATIVE ANALYSIS PLOT
# ============================================================

plt.figure(figsize=(12, 6))

plt.bar(
    results_df["Class"][:-1],
    results_df["FN"][:-1]
)

plt.xticks(rotation=30)

plt.ylabel("False Negatives")

plt.title("Class-wise False Negatives")

fn_path = os.path.join(
    SAVE_DIR,
    "false_negative_analysis.png"
)

plt.savefig(
    fn_path,
    dpi=600,
    bbox_inches="tight"
)

plt.close()

print("\nSaved FN analysis plot.")

# ============================================================
# MACRO SUMMARY TXT
# ============================================================

summary_path = os.path.join(
    SAVE_DIR,
    "macro_results.txt"
)

with open(summary_path, "w") as f:

    f.write("================================================\n")
    f.write("Macro Results\n")
    f.write("================================================\n\n")

    f.write(f"Precision     : {macro_precision:.4f}\n")
    f.write(f"Recall        : {macro_recall:.4f}\n")
    f.write(f"F1-Score      : {macro_f1:.4f}\n")
    f.write(f"Sensitivity   : {macro_sensitivity:.4f}\n")
    f.write(f"Specificity   : {macro_specificity:.4f}\n")
    f.write(f"mAP50         : {macro_map50:.4f}\n")
    f.write(f"mAP50-95      : {macro_map5095:.4f}\n")

print("\nSaved macro summary.")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n================================================")
print("FINAL MACRO RESULTS")
print("================================================")

print(f"Precision     : {macro_precision:.4f}")
print(f"Recall        : {macro_recall:.4f}")
print(f"F1-Score      : {macro_f1:.4f}")
print(f"Sensitivity   : {macro_sensitivity:.4f}")
print(f"Specificity   : {macro_specificity:.4f}")
print(f"mAP50         : {macro_map50:.4f}")
print(f"mAP50-95      : {macro_map5095:.4f}")

print("\n================================================")
print("ALL RESULTS SAVED")
print("================================================")

Ultralytics 8.4.51  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
YOLOv8m summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 481.2139.5 MB/s, size: 45.5 KB)
val: Scanning E:\Mahin\Thyroid Project\Thyroid_Dataset\final_3class_dataset\test\labels.cache... 99 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 99/99  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.7it/s 4.9s0.2s
                   all         99        507      0.718      0.659      0.639      0.456
        Nuclear groove         89        412      0.611      0.662      0.657      0.471
Nuclear pseudoinclusions         14         21      0.907      0.571      0.616      0.462
Orphan Annie Eye nuclei         27         74      0.636      0.743      0.644      0.435
Speed: 1.7ms preprocess, 13.3ms inference, 0.0ms loss, 3.5ms postprocess per image
Saving E:\

In [1]:
# ============================================================
# COMPLETE TEST EVALUATION PIPELINE
# TP / FP / FN ANALYSIS + FN RATIO
# 600 DPI CONFUSION MATRIX
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

from sklearn.metrics import (
    ConfusionMatrixDisplay
)

# ============================================================
# PATHS
# ============================================================



SAVE_DIR = str(OUTPUT_ROOT / "evaluation_2")

os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(MODEL_PATH)

# ============================================================
# RUN TEST VALIDATION
# ============================================================

metrics = model.val(
    data=DATA_YAML,
    split="test",

    imgsz=640,
    batch=8,

    conf=0.25,
    iou=0.5,

    device=0,

    save_json=True,
    plots=True,

    project=SAVE_DIR,
    name="test_eval"
)

# ============================================================
# CLASS NAMES
# ============================================================

class_names = model.names

num_classes = len(class_names)

print("\nClasses:")
print(class_names)

# ============================================================
# YOLO METRICS
# ============================================================

precision_per_class = metrics.box.p
recall_per_class    = metrics.box.r

map50_per_class     = metrics.box.ap50
map5095_per_class   = metrics.box.ap

# ============================================================
# F1 SCORE
# ============================================================

f1_per_class = (
    2
    * precision_per_class
    * recall_per_class
) / (
    precision_per_class
    + recall_per_class
    + 1e-9
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = metrics.confusion_matrix.matrix

cm = cm[:num_classes, :num_classes]

# ============================================================
# TP / FP / FN
# ============================================================

tp_list = []
fp_list = []
fn_list = []

fn_ratio_list = []
fp_ratio_list = []

for i in range(num_classes):

    TP = cm[i, i]

    FN = cm[i, :].sum() - TP

    FP = cm[:, i].sum() - TP

    fn_ratio = FN / (TP + FN + 1e-9)

    fp_ratio = FP / (TP + FP + 1e-9)

    tp_list.append(TP)
    fp_list.append(FP)
    fn_list.append(FN)

    fn_ratio_list.append(fn_ratio)
    fp_ratio_list.append(fp_ratio)

# ============================================================
# MACRO METRICS
# ============================================================

macro_precision = np.mean(precision_per_class)
macro_recall    = np.mean(recall_per_class)

macro_f1 = np.mean(f1_per_class)

macro_map50   = np.mean(map50_per_class)
macro_map5095 = np.mean(map5095_per_class)

macro_fn_ratio = np.mean(fn_ratio_list)
macro_fp_ratio = np.mean(fp_ratio_list)

# ============================================================
# RESULTS TABLE
# ============================================================

results_df = pd.DataFrame({

    "Class": [
        class_names[i]
        for i in range(num_classes)
    ],

    "Precision": precision_per_class,
    "Recall": recall_per_class,
    "F1-Score": f1_per_class,

    "mAP50": map50_per_class,
    "mAP50-95": map5095_per_class,

    "TP": tp_list,
    "FP": fp_list,
    "FN": fn_list,

    "FN Ratio": fn_ratio_list,
    "FP Ratio": fp_ratio_list
})

results_df = results_df.round(4)

# ============================================================
# ADD MACRO ROW
# ============================================================

macro_row = pd.DataFrame([{

    "Class": "Macro Average",

    "Precision": round(macro_precision, 4),
    "Recall": round(macro_recall, 4),
    "F1-Score": round(macro_f1, 4),

    "mAP50": round(macro_map50, 4),
    "mAP50-95": round(macro_map5095, 4),

    "TP": np.sum(tp_list),
    "FP": np.sum(fp_list),
    "FN": np.sum(fn_list),

    "FN Ratio": round(macro_fn_ratio, 4),
    "FP Ratio": round(macro_fp_ratio, 4)
}])

results_df = pd.concat(
    [results_df, macro_row],
    ignore_index=True
)

# ============================================================
# SAVE CSV
# ============================================================

csv_path = os.path.join(
    SAVE_DIR,
    "classwise_metrics.csv"
)

results_df.to_csv(
    csv_path,
    index=False
)

print("\nSaved metrics CSV:")
print(csv_path)

# ============================================================
# SAVE TP / FP / FN SUMMARY
# ============================================================

error_summary_df = results_df[
    [
        "Class",
        "TP",
        "FP",
        "FN",
        "FN Ratio",
        "FP Ratio"
    ]
]

error_summary_path = os.path.join(
    SAVE_DIR,
    "tp_fp_fn_summary.csv"
)

error_summary_df.to_csv(
    error_summary_path,
    index=False
)

print("\nSaved TP / FP / FN summary.")

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n================================================")
print("TEST RESULTS")
print("================================================\n")

print(results_df)

# ============================================================
# 600 DPI CONFUSION MATRIX
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        class_names[i]
        for i in range(num_classes)
    ]
)

disp.plot(
    cmap="Blues",
    ax=ax,
    colorbar=False
)

plt.title(
    "Thyroid Abnormality Detection Confusion Matrix"
)

cm_path = os.path.join(
    SAVE_DIR,
    "confusion_matrix.png"
)

plt.savefig(
    cm_path,
    dpi=600,
    bbox_inches="tight"
)

plt.close()

print("\nSaved confusion matrix.")

# ============================================================
# FN RATIO PLOT
# ============================================================

plt.figure(figsize=(12, 6))

plt.bar(
    results_df["Class"][:-1],
    results_df["FN Ratio"][:-1]
)

plt.xticks(rotation=30)

plt.ylabel("FN Ratio")

plt.title(
    "Class-wise False Negative Ratio"
)

fn_ratio_path = os.path.join(
    SAVE_DIR,
    "fn_ratio_analysis.png"
)

plt.savefig(
    fn_ratio_path,
    dpi=600,
    bbox_inches="tight"
)

plt.close()

print("\nSaved FN ratio analysis plot.")

# ============================================================
# TP / FP / FN GROUPED BAR PLOT
# ============================================================

classes = results_df["Class"][:-1]

x = np.arange(len(classes))

width = 0.25

plt.figure(figsize=(14, 6))

plt.bar(
    x - width,
    results_df["TP"][:-1],
    width,
    label="TP"
)

plt.bar(
    x,
    results_df["FP"][:-1],
    width,
    label="FP"
)

plt.bar(
    x + width,
    results_df["FN"][:-1],
    width,
    label="FN"
)

plt.xticks(
    x,
    classes,
    rotation=30
)

plt.ylabel("Count")

plt.title(
    "TP / FP / FN Error Distribution"
)

plt.legend()

grouped_plot_path = os.path.join(
    SAVE_DIR,
    "tp_fp_fn_error_distribution.png"
)

plt.savefig(
    grouped_plot_path,
    dpi=600,
    bbox_inches="tight"
)

plt.close()

print("\nSaved TP / FP / FN grouped plot.")

# ============================================================
# MACRO SUMMARY TXT
# ============================================================

summary_path = os.path.join(
    SAVE_DIR,
    "macro_results.txt"
)

with open(summary_path, "w") as f:

    f.write("================================================\n")
    f.write("Macro Results\n")
    f.write("================================================\n\n")

    f.write(f"Precision     : {macro_precision:.4f}\n")
    f.write(f"Recall        : {macro_recall:.4f}\n")
    f.write(f"F1-Score      : {macro_f1:.4f}\n")
    f.write(f"mAP50         : {macro_map50:.4f}\n")
    f.write(f"mAP50-95      : {macro_map5095:.4f}\n")
    f.write(f"FN Ratio      : {macro_fn_ratio:.4f}\n")
    f.write(f"FP Ratio      : {macro_fp_ratio:.4f}\n")

print("\nSaved macro summary.")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n================================================")
print("FINAL MACRO RESULTS")
print("================================================")

print(f"Precision     : {macro_precision:.4f}")
print(f"Recall        : {macro_recall:.4f}")
print(f"F1-Score      : {macro_f1:.4f}")
print(f"mAP50         : {macro_map50:.4f}")
print(f"mAP50-95      : {macro_map5095:.4f}")
print(f"FN Ratio      : {macro_fn_ratio:.4f}")
print(f"FP Ratio      : {macro_fp_ratio:.4f}")

print("\n================================================")
print("ALL RESULTS SAVED")
print("================================================")

Ultralytics 8.4.51  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
YOLOv8m summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 6.41.4 MB/s, size: 51.7 KB)
val: Scanning E:\Mahin\Thyroid Project\Thyroid_Dataset\final_3class_dataset\test\labels.cache... 99 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 99/99  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.7it/s 4.8s0.1s
                   all         99        507      0.709      0.662      0.639      0.456
        Nuclear groove         89        412      0.611       0.67      0.657      0.471
Nuclear pseudoinclusions         14         21      0.904      0.571      0.616      0.462
Orphan Annie Eye nuclei         27         74       0.61      0.743      0.644      0.435
Speed: 1.7ms preprocess, 10.4ms inference, 0.0ms loss, 5.1ms postprocess per image
Saving E:\Mahi